In [1]:
import os
import json
import numpy as np
from PIL import Image
from tqdm import tqdm
from collections import defaultdict

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim import AdamW

class SpeedEstimationModel2(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(15, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 + 15, 128),  
            nn.LeakyReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, imgs, speeds):
        x = imgs.squeeze(2)
        x = self.cnn(x)
        x = x.view(x.size(0), -1)
        x = torch.cat([x, speeds], dim=1)
        return self.fc(x).squeeze(1)

class SpeedDataset(Dataset):
    def __init__(self, crop_root, annot_root):
        self.items = []
        self.transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor()
        ])
        scene_ids = sorted(os.listdir(crop_root))
        for sid in scene_ids:
            if int(sid) > 240: continue
            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path): continue
            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")])
            if len(files) == 0: continue
            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)
            min_len = min(len(files), len(ann['sequence']))
            if min_len == 0: continue

            speeds_all = np.array([ann['sequence'][i]['OwnSpeed'] / 3.6 for i in range(min_len)], dtype=np.float32)
            tgt_speeds_all = np.array([ann['sequence'][i]['TgtSpeed_ref'] / 3.6 for i in range(min_len)], dtype=np.float32)
            rel_speeds_all = tgt_speeds_all - speeds_all

            for i in range(max(1, min_len - 14)):
                img_paths = [os.path.join(crop_dir, files[j]) for j in range(i, min(i + 15, min_len))]
                if len(img_paths) < 15:
                    img_paths += [img_paths[-1]] * (15 - len(img_paths))
                speeds = speeds_all[i:i+15]
                if len(speeds) < 15:
                    speeds = np.pad(speeds, (0, 15 - len(speeds)), mode='edge')
                tgt_rel_speed = np.mean(rel_speeds_all[i:i+15])
                self.items.append((img_paths, speeds, tgt_rel_speed, sid))

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        img_paths, speeds, tgt_rel_speed, scene_id = self.items[idx]
        imgs = []
        for p in img_paths:
            try:
                img = Image.open(p).convert("L")
                imgs.append(self.transform(img))
            except:
                imgs.append(torch.zeros((1, 64, 64)))
        imgs = torch.stack(imgs)
        return imgs, torch.tensor(speeds), torch.tensor(tgt_rel_speed), scene_id

def train_speed_model2():
    crop_dir = "disparity_crops"
    annot_dir = "train_annotations"
    test_annot_dir = "test_annotations"

    full_dataset = SpeedDataset(crop_dir, annot_dir)
    train_dataset = torch.utils.data.Subset(full_dataset, range(min(len(full_dataset), 500)))

    def collate_fn(batch):
        imgs, speeds, tgts, sids = zip(*batch)
        return torch.stack(imgs), torch.stack(speeds), torch.tensor(tgts), list(sids)

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)
    inference_loader = DataLoader(full_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SpeedEstimationModel2().to(device)
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    loss_fn = nn.MSELoss()

    for epoch in range(50):
        model.train()
        total_loss = 0
        for imgs, speeds, tgts, _ in tqdm(train_loader, desc=f"[Epoch {epoch+1}]"):
            imgs, speeds, tgts = imgs.to(device), speeds.to(device), tgts.to(device)
            preds = model(imgs, speeds)
            loss = loss_fn(preds, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * imgs.size(0)
        print(f"Epoch {epoch+1} - Train MSE Loss (rel): {total_loss / len(train_dataset):.4f}")

    torch.save(model.state_dict(), "speed_model2_rel_nodist.pth")
    print(" speed_model2_rel_nodist.pth")

  
    model.eval()
    result = defaultdict(list)
    with torch.no_grad():
        for imgs, speeds, _, scene_ids in tqdm(inference_loader, desc="推論中"):
            imgs, speeds = imgs.to(device), speeds.to(device)
            pred_rel = model(imgs, speeds)
            pred_abs = (pred_rel + speeds[:, -1]).cpu().numpy() * 3.6  
            for pred, sid in zip(pred_abs.tolist(), scene_ids):
                result[sid].append(round(pred, 2))

    final_result = {}
    for i in range(241):
        sid = f"{i:03d}"
        annot_path = os.path.join(test_annot_dir, f"{sid}.json")
        if not os.path.exists(annot_path):
            final_result[sid] = []
            continue
        with open(annot_path, encoding="utf-8") as f:
            ann = json.load(f)
        seq_len = len(ann["sequence"])
        preds = result.get(sid, [])
        if len(preds) < seq_len:
            preds += [preds[-1] if preds else 0.0] * (seq_len - len(preds))
        final_result[sid] = preds[:seq_len]

    with open("submission.json", "w", encoding="utf-8") as f:
        json.dump(final_result, f, indent=2, ensure_ascii=False)
    print("submission.json")

if __name__ == "__main__":
    train_speed_model2()


[Epoch 1]: 100%|██████████| 125/125 [00:04<00:00, 28.94it/s]


Epoch 1 - Train MSE Loss (rel): 2.5041


[Epoch 2]: 100%|██████████| 125/125 [00:03<00:00, 41.31it/s]


Epoch 2 - Train MSE Loss (rel): 1.7680


[Epoch 3]: 100%|██████████| 125/125 [00:03<00:00, 40.46it/s]


...

Epoch 49 - Train MSE Loss (rel): 0.3637


[Epoch 50]: 100%|██████████| 125/125 [00:03<00:00, 38.63it/s]


Epoch 50 - Train MSE Loss (rel): 0.3472
 speed_model2_rel_nodist.pth


推論中: 100%|██████████| 6343/6343 [02:37<00:00, 40.29it/s]


submission.json
